Notebook criado para recuperar os arquivos em csv na pasta raw, transformar em parquet e enviar para a pasta bronze do AWS (e local) 

In [1]:
%load_ext autoreload
%autoreload 2


from IPython import get_ipython
import pandas as pd 
import os
import dotenv
from pathlib import Path
from pandas import DataFrame
from botocore.exceptions import ClientError

# Instala o boto3 biblioteca utilizada para conexão do aws 
try:
    import boto3
except ModuleNotFoundError:
    get_ipython().run_line_magic('pip', 'install boto3')
    import boto3

dotenv.load_dotenv()

# 2. Trata o pacote local 'src' para conseguir extrair as funções uteis
try:
    from src.data.utils import gerar_df_dic, converter_para_parquet_bytes, salvar_parquet_local
except ModuleNotFoundError:
    # Se falhar, instala o pacote local e importa
    get_ipython().run_line_magic('pip', 'install -e ..')
    from src.data.utils import gerar_df_dic, converter_para_parquet_bytes, salvar_parquet_local

In [2]:
# Recuperar as variaveis para acessar o AWS 
region_name = os.getenv("region_name")
aws_access_key_id=os.getenv("aws_access_key_id")
aws_secret_access_key=os.getenv("aws_secret_access_key")
aws_session_token=os.getenv("aws_session_token")

# Gerar conexão com o S3 
boto3.setup_default_session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,  # <--- Não esqueça de pegar este lá no "AWS Details"
    region_name=region_name
)

configurado o boto 3 iremos utiliza-lo para acessar o nosso AWS atraves das nossas variaveis, iremos verificar se já existe o bucket da fiap-pos-tech-002, caso nao exista iremos cria-lo, se existir só iremos conectar a ele

In [3]:
s3 = boto3.client('s3')
nome_do_bucket = 'fiap-tech-challenge-002'

response = s3.list_buckets()
buckets_existentes = [bucket['Name'] for bucket in response['Buckets']]

try:
    # verifica a existencia do bucket da Fiap na AWS conectada
    s3.head_bucket(Bucket=nome_do_bucket)
    print(f"Bucket '{nome_do_bucket}' já existe.")
except ClientError as e:
    # Se der erro 404 (Not Found), significa que o bucket não existe então iremos cria-lo
    error_code = e.response['Error']['Code']
    if error_code == '404':
        s3.create_bucket(Bucket=nome_do_bucket)
        print(f"Bucket '{nome_do_bucket}' criado com sucesso.")
    else:
        # Outro erro
        print(f"Erro ao acessar o bucket: {e}")

Bucket 'fiap-tech-challenge-002' já existe.


Utilizamos a função gerar_df_dic localizada no src\data\utils.py 
a função é utilizada para carregar os csv no Raw e carregar todos os dataframes para que possamos transforma-los em .parquet

In [4]:
# ano = 2023 
# raw_aluno, dicionario_aluno = gerar_df_dic(ano, 'TS_ALUNO')
# raw_itens, dicionario_itens = gerar_df_dic(ano,'TS_ITEM')
# raw_uf, dicionario_uf = gerar_df_dic(ano,'TS_ESTADO')
# raw_municipio, dicionario_municipio = gerar_df_dic(ano,'TS_MUNICIPIO')

In [ ]:
ano = 2023 
tabelas = ['TS_ALUNO','TS_ITEM','TS_ESTADO','TS_MUNICIPIO']

for tabela in tabelas:
    print(f'trabalhando com a tabela  {tabela}')
    caminho_df = Path(f"../data/bronze/ano={ano}/dados/{tabela}.parquet")
    caminho_dicionario =  Path(f"../data/bronze/ano={ano}/dicionario/dicionario_{tabela}.parquet")

    df, dic = gerar_df_dic(ano, tabela)
    print(f'gerando parquets  {tabela}')
   # df_parquet = converter_para_parquet_bytes(df)
   # dic_parquet = converter_para_parquet_bytes(dic)
    print(f'salvando  {tabela}')
    salvar_parquet_local(df, caminho_df)
    print(f'Sucesso ao salvar {tabela} em {caminho_df}')
    salvar_parquet_local(dic, caminho_dicionario)
    print(f'Sucesso ao gerar o dicionario {tabela} em {caminho_dicionario}')

trabalhando com a tabela  TS_ALUNO
Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx
gerando parquets  TS_ALUNO
salvando  TS_ALUNO
Arquivo salvo localmente em: ..\data\bronze\ano=2023\TS_ALUNO.parquet
Sucesso ao salvar TS_ALUNO em ..\data\bronze\ano=2023\TS_ALUNO.parquet
Arquivo salvo localmente em: ..\data\bronze\ano=2023\dicionario_TS_ALUNO.parquet
Sucesso ao gerar o dicionario TS_ALUNO em ..\data\bronze\ano=2023\dicionario_TS_ALUNO.parquet
trabalhando com a tabela  TS_ITEM
Lendo dicionário em: ..\data\raw\dados_2023\DICIONÁRIO\Dicionario_Microdados_AEEB_2023.xlsx
gerando parquets  TS_ITEM
salvando  TS_ITEM
Arquivo salvo localmente em: ..\data\bronze\ano=2023\TS_ITEM.parquet
Sucesso ao salvar TS_ITEM em ..\data\bronze\ano=2023\TS_ITEM.parquet
Arquivo salvo localmente em: ..\data\bronze\ano=2023\dicionario_TS_ITEM.parquet
Sucesso ao gerar o dicionario TS_ITEM em ..\data\bronze\ano=2023\dicionario_TS_ITEM.parquet
trabalhando com a tabela  TS_ES